# Phân chia tập dữ liệu DurianLDD (Train / Val / Test)

Notebook này thực hiện toàn bộ quy trình kiểm tra dữ liệu, phân chia tập dữ liệu DurianLDD (4.437 ảnh) thành ba tập train (70%), validation (10%) và test (20%) có kiểm soát chống rò rỉ (leakage) do các nhóm ảnh gần trùng, kiểm chứng nghiêm ngặt tính tái lập và lưu manifest khóa test.

---
## 0. Cấu hình thực nghiệm (Config)

**Mục đích:** Xác định `PROJECT_ROOT` linh hoạt (không hardcode) và đọc toàn bộ hằng số thực nghiệm (seed ngẫu nhiên, đường dẫn tập tin dữ liệu, tỉ lệ phân chia train/val/test, số fold chia tập, ngưỡng dung sai kiểm chứng, số lượng ảnh kỳ vọng tổng thể và theo từng lớp, ngưỡng nhóm ảnh gần trùng) từ tệp cấu hình tập trung `configs/split.yaml`.

In [ ]:
import sys
from pathlib import Path
import yaml

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "configs" / "split.yaml").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án chứa configs/split.yaml!")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "split.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

SEED = config["seed"]

INVENTORY_PATH = PROJECT_ROOT / config["inventory_path"]
NEAR_DUP_PATH = PROJECT_ROOT / config["near_dup_path"]
MANIFESTS_DIR = PROJECT_ROOT / config["manifests_dir"]
MANIFEST_PATH = PROJECT_ROOT / config["manifest_path"]
META_PATH = PROJECT_ROOT / config["meta_path"]
PROTOCOL_PATH = PROJECT_ROOT / config["protocol_path"]

TRAIN_RATIO = config["train_ratio"]
VAL_RATIO = config["val_ratio"]
TEST_RATIO = config["test_ratio"]

N_SPLITS_TEST = config["n_splits_test"]
N_SPLITS_VAL = config["n_splits_val"]

MAX_SPLIT_RATIO_TOLERANCE = config["max_split_ratio_tolerance"]
MAX_CLASS_RATIO_TOLERANCE = config["max_class_ratio_tolerance"]

EXPECTED_TOTAL_IMAGES = config["expected_total_images"]
EXPECTED_CLASS_COUNTS = config["expected_class_counts"]

EXPECTED_NEAR_DUP_GROUPS = config["expected_near_dup_groups"]
EXPECTED_NEAR_DUP_IMAGES = config["expected_near_dup_images"]
MAX_HAMMING_DISTANCE = config["max_hamming_distance"]


In [ ]:
assert round(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 5) == 1.0, (
    f"Tổng các tỉ lệ phân chia phải bằng 1.0 (hiện tại: {TRAIN_RATIO + VAL_RATIO + TEST_RATIO})"
)
assert sum(EXPECTED_CLASS_COUNTS.values()) == EXPECTED_TOTAL_IMAGES, (
    f"Tổng số ảnh trong EXPECTED_CLASS_COUNTS ({sum(EXPECTED_CLASS_COUNTS.values())}) "
    f"không khớp với EXPECTED_TOTAL_IMAGES ({EXPECTED_TOTAL_IMAGES})"
)
assert INVENTORY_PATH.is_file(), f"Không tìm thấy file inventory tại: {INVENTORY_PATH}"
assert NEAR_DUP_PATH.is_file(), f"Không tìm thấy file near duplicates tại: {NEAR_DUP_PATH}"
assert MANIFESTS_DIR.is_dir(), f"Không tìm thấy thư mục manifests tại: {MANIFESTS_DIR}"
assert PROTOCOL_PATH.is_file(), f"Không tìm thấy file protocol tại: {PROTOCOL_PATH}"


In [ ]:
print("=" * 68)
print("  CẤU HÌNH THỰC NGHIỆM CHIA TẬP DURIANLDD (Đọc từ configs/split.yaml)")
print("=" * 68)
print(f"• Random SEED              : {SEED}")
print(f"• PROJECT_ROOT             : {PROJECT_ROOT}")
print(f"• CONFIG_PATH              : {CONFIG_PATH.relative_to(PROJECT_ROOT)}")
print(f"• INVENTORY_PATH           : {INVENTORY_PATH.relative_to(PROJECT_ROOT)}")
print(f"• NEAR_DUP_PATH            : {NEAR_DUP_PATH.relative_to(PROJECT_ROOT)}")
print(f"• MANIFEST_PATH            : {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
print(f"• META_PATH                : {META_PATH.relative_to(PROJECT_ROOT)}")
print(f"• PROTOCOL_PATH            : {PROTOCOL_PATH.relative_to(PROJECT_ROOT)}")
print(f"• Tỉ lệ split (Train/Val/Test): {TRAIN_RATIO*100:.0f}% / {VAL_RATIO*100:.0f}% / {TEST_RATIO*100:.0f}%")
print(f"• Dung sai kiểm chứng      : split <= {MAX_SPLIT_RATIO_TOLERANCE*100:.0f}%, class <= {MAX_CLASS_RATIO_TOLERANCE*100:.0f}%")
print(f"• Folds chia tập           : N_SPLITS_TEST={N_SPLITS_TEST}, N_SPLITS_VAL={N_SPLITS_VAL}")
print(f"• Tổng số ảnh kỳ vọng      : {EXPECTED_TOTAL_IMAGES}")
print(f"• Phân bố theo lớp ({len(EXPECTED_CLASS_COUNTS)} lớp):")
for class_name, count in EXPECTED_CLASS_COUNTS.items():
    pct = (count / EXPECTED_TOTAL_IMAGES) * 100
    print(f"    - {class_name:<21}: {count:>4} ảnh ({pct:5.2f}%)")
print(f"• Nhóm gần trùng kỳ vọng   : {EXPECTED_NEAR_DUP_GROUPS} nhóm ({EXPECTED_NEAR_DUP_IMAGES} ảnh, Hamming <= {MAX_HAMMING_DISTANCE})")
print("=" * 68)
print("✓ Tất cả cấu hình từ YAML và đường dẫn đã được xác thực hợp lệ.")


  CẤU HÌNH THỰC NGHIỆM CHIA TẬP DURIANLDD (Đọc từ configs/split.yaml)
• Random SEED              : 42
• PROJECT_ROOT             : C:\WorkSpace\Study\durian-leaf-disease-robustness
• CONFIG_PATH              : configs\split.yaml
• INVENTORY_PATH           : reports\data_quality\tables\image_inventory.csv
• NEAR_DUP_PATH            : reports\data_quality\tables\near_duplicate_groups.csv
• MANIFEST_PATH            : manifests\split_manifest.csv
• META_PATH                : manifests\split_meta.json
• PROTOCOL_PATH            : manifests\SPLIT_PROTOCOL.md
• Tỉ lệ split (Train/Val/Test): 70% / 10% / 20%
• Dung sai kiểm chứng      : split <= 1%, class <= 2%
• Folds chia tập           : N_SPLITS_TEST=5, N_SPLITS_VAL=8
• Tổng số ảnh kỳ vọng      : 4437
• Phân bố theo lớp (5 lớp):
    - ALGAL_LEAF_SPOT      :  733 ảnh (16.52%)
    - ALLOCARIDARA_ATTACK  :  913 ảnh (20.58%)
    - HEALTHY_LEAF         :  976 ảnh (22.00%)
    - LEAF_BLIGHT          :  937 ảnh (21.12%)
    - PHOMOPSIS_LEAF_SPOT  :

---
## 1. Kiểm tra dữ liệu

**Mục đích:** 
1. Đọc tệp thống kê dữ liệu `reports/data_quality/tables/image_inventory.csv`, in danh sách cột thật và 5 dòng đầu.
2. Lọc chỉ giữ các ảnh thuộc DurianLDD (sử dụng cột `relative_path`), loại bỏ hoàn toàn Vietnamese durian leaf.
3. Assert kiểm tra tính toàn vẹn: đúng 4.437 ảnh, số ảnh mỗi lớp khớp tuyệt đối với Config, không trùng lặp đường dẫn (`relative_path`).
4. Dựa vào cột `group_id`: assert có đúng 4 nhóm gần trùng (tổng 9 ảnh, Hamming ≤ 5). In bảng chi tiết từng nhóm và kiểm tra mỗi nhóm đều có nhãn lớp đồng nhất.
5. Gán cột `group_key` đại diện đơn vị chia tập: các ảnh trong nhóm gần trùng dùng chung `group_key`, các ảnh không thuộc nhóm được gán `group_key` riêng biệt (1 ảnh/nhóm).

In [ ]:
import pandas as pd

df_raw = pd.read_csv(INVENTORY_PATH)

print(f"Tổng số dòng đọc được từ inventory: {len(df_raw)}")
print(f"Danh sách 24 cột thật trong image_inventory.csv:\n{df_raw.columns.tolist()}\n")
print("5 dòng đầu tiên:")
print(df_raw.head(5))


Tổng số dòng đọc được từ inventory: 4437
Danh sách 24 cột thật trong image_inventory.csv:
['relative_path', 'split', 'class_name', 'source_id', 'file_bytes', 'sha256', 'width', 'height', 'aspect_ratio', 'mode', 'brightness', 'contrast', 'sharpness', 'saturation', 'high_frequency', 'pixel_sha256', 'flip_pixel_sha256', 'dhash', 'dhash_flip', 'exif_datetime', 'readable', 'error', 'group_id', 'group_size']

5 dòng đầu tiên:
                                       relative_path  ... group_size
0  DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT...  ...          1
1  DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT...  ...          1
2  DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT...  ...          1
3  DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT...  ...          1
4  DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT...  ...          1

[5 rows x 24 columns]


In [ ]:
df_durian = df_raw[
    df_raw["relative_path"].str.startswith("DLD_FinalDataset_224_spit")
    & ~df_raw["relative_path"].str.lower().str.contains("vietnamese")
].copy()

df_durian = df_durian.reset_index(drop=True)
print(f"Số ảnh DurianLDD sau khi lọc: {len(df_durian)}")


Số ảnh DurianLDD sau khi lọc: 4437


In [ ]:
assert len(df_durian) == EXPECTED_TOTAL_IMAGES, (
    f"Tổng số ảnh ({len(df_durian)}) không khớp kỳ vọng ({EXPECTED_TOTAL_IMAGES})!"
)

assert df_durian["relative_path"].is_unique, "Phát hiện relative_path bị trùng lặp!"

actual_class_counts = df_durian["class_name"].value_counts().to_dict()
for class_name, expected_count in EXPECTED_CLASS_COUNTS.items():
    actual_count = actual_class_counts.get(class_name, 0)
    assert actual_count == expected_count, (
        f"Lớp '{class_name}': thực tế {actual_count} ảnh, kỳ vọng {expected_count} ảnh!"
    )

assert set(actual_class_counts.keys()) == set(EXPECTED_CLASS_COUNTS.keys()), (
    f"Tập hợp tên lớp không khớp! Thực tế: {set(actual_class_counts.keys())}, Kỳ vọng: {set(EXPECTED_CLASS_COUNTS.keys())}"
)

print("✓ Assert thành công: Đúng 4.437 ảnh, đúng 5 lớp nguyên bản và relative_path không trùng lặp.")


✓ Assert thành công: Đúng 4.437 ảnh, đúng 5 lớp nguyên bản và relative_path không trùng lặp.


In [ ]:
group_counts = df_durian["group_id"].value_counts()
dup_group_ids = sorted(group_counts[group_counts > 1].index.tolist())

df_near_dups = df_durian[df_durian["group_id"].isin(dup_group_ids)].copy()

assert len(dup_group_ids) == EXPECTED_NEAR_DUP_GROUPS, (
    f"Số nhóm gần trùng ({len(dup_group_ids)}) != {EXPECTED_NEAR_DUP_GROUPS}!"
)
assert len(df_near_dups) == EXPECTED_NEAR_DUP_IMAGES, (
    f"Tổng số ảnh trong nhóm gần trùng ({len(df_near_dups)}) != {EXPECTED_NEAR_DUP_IMAGES}!"
)

for gid in dup_group_ids:
    classes_in_group = df_near_dups[df_near_dups["group_id"] == gid]["class_name"].unique()
    assert len(classes_in_group) == 1, (
        f"Nhóm group_id={gid} chứa nhiều lớp khác nhau: {classes_in_group}!"
    )

print(f"Xác nhận có đúng {len(dup_group_ids)} nhóm gần trùng (tổng {len(df_near_dups)} ảnh, Hamming <= {MAX_HAMMING_DISTANCE}):\n")
dup_table = df_near_dups[["group_id", "relative_path", "class_name"]].sort_values(by=["group_id", "relative_path"])
print(dup_table.to_string(index=False))


Xác nhận có đúng 4 nhóm gần trùng (tổng 9 ảnh, Hamming <= 5):

 group_id                                                        relative_path          class_name
       26      DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT/to_label_197.jpg     ALGAL_LEAF_SPOT
       26     DLD_FinalDataset_224_spit/train/ALGAL_LEAF_SPOT/to_label_196.jpg     ALGAL_LEAF_SPOT
      517        DLD_FinalDataset_224_spit/test/HEALTHY_LEAF/to_label_2663.jpg        HEALTHY_LEAF
      517       DLD_FinalDataset_224_spit/train/HEALTHY_LEAF/to_label_2682.jpg        HEALTHY_LEAF
      517       DLD_FinalDataset_224_spit/train/HEALTHY_LEAF/to_label_2683.jpg        HEALTHY_LEAF
      632         DLD_FinalDataset_224_spit/test/LEAF_BLIGHT/to_label_3212.jpg         LEAF_BLIGHT
      632         DLD_FinalDataset_224_spit/test/LEAF_BLIGHT/to_label_3222.jpg         LEAF_BLIGHT
     1980 DLD_FinalDataset_224_spit/train/ALLOCARIDARA_ATTACK/to_label_909.jpg ALLOCARIDARA_ATTACK
     1980 DLD_FinalDataset_224_spit/train/ALLO

In [ ]:
df_durian["group_key"] = df_durian.apply(
    lambda row: f"group_{int(row['group_id'])}" if row["group_id"] in dup_group_ids else f"single_{row.name}",
    axis=1,
)

key_counts = df_durian["group_key"].value_counts()
single_keys = key_counts[key_counts == 1]
multi_keys = key_counts[key_counts > 1]

assert len(multi_keys) == EXPECTED_NEAR_DUP_GROUPS, (
    f"Số lượng multi-image group_key ({len(multi_keys)}) != {EXPECTED_NEAR_DUP_GROUPS}!"
)
assert multi_keys.sum() == EXPECTED_NEAR_DUP_IMAGES, (
    f"Tổng số ảnh trong multi-image group_key ({multi_keys.sum()}) != {EXPECTED_NEAR_DUP_IMAGES}!"
)
assert len(single_keys) == EXPECTED_TOTAL_IMAGES - EXPECTED_NEAR_DUP_IMAGES, (
    f"Số key đơn lẻ ({len(single_keys)}) != {EXPECTED_TOTAL_IMAGES - EXPECTED_NEAR_DUP_IMAGES}!"
)
assert df_durian["group_key"].nunique() == len(single_keys) + len(multi_keys)

print("✓ Gán group_key thành công:")
print(f"  • Số nhóm gần trùng (đa ảnh) : {len(multi_keys)} nhóm ({multi_keys.sum()} ảnh)")
print(f"  • Số nhóm đơn lẻ (1 ảnh/key) : {len(single_keys)} nhóm ({len(single_keys)} ảnh)")
print(f"  • Tổng số group_key duy nhất : {df_durian['group_key'].nunique()} key đại diện")


✓ Gán group_key thành công:
  • Số nhóm gần trùng (đa ảnh) : 4 nhóm (9 ảnh)
  • Số nhóm đơn lẻ (1 ảnh/key) : 4428 nhóm (4428 ảnh)
  • Tổng số group_key duy nhất : 4432 key đại diện


In [ ]:
print("=" * 70)
print("         KẾT QUẢ KIỂM TRA DỮ LIỆU DURIANLDD (PHẦN 1)")
print("=" * 70)
print(f"1. Tổng số ảnh DurianLDD       : {len(df_durian)} / {EXPECTED_TOTAL_IMAGES} (Khớp 100%)")
print(f"2. Tính duy nhất relative_path : Duy nhất 100% ({df_durian['relative_path'].nunique()} path)")
print(f"3. Phân bố 5 lớp bệnh:")
for cname, expected_c in EXPECTED_CLASS_COUNTS.items():
    actual_c = actual_class_counts[cname]
    print(f"   - {cname:<21}: {actual_c:>4} ảnh (kỳ vọng: {expected_c:>4}) -> KHỚP")
print(f"4. Nhóm gần trùng (Hamming <= 5): {len(dup_group_ids)} nhóm, tổng {len(df_near_dups)} ảnh")
for gid in dup_group_ids:
    g_sub = df_near_dups[df_near_dups["group_id"] == gid]
    print(f"   - Nhóm group_id={gid:<4}: {len(g_sub)} ảnh, lớp {g_sub['class_name'].iloc[0]}")
print(f"   -> Kiểm tra đồng nhất lớp   : 100% mỗi nhóm chỉ thuộc 1 lớp duy nhất")
print(f"5. Đơn vị chia tập (group_key)  : {df_durian['group_key'].nunique()} key sẵn sàng cho split")
print("=" * 70)
print("✓ Toàn bộ kiểm tra dữ liệu đạt yêu cầu. Sẵn sàng sang bước chia tập.")


         KẾT QUẢ KIỂM TRA DỮ LIỆU DURIANLDD (PHẦN 1)
1. Tổng số ảnh DurianLDD       : 4437 / 4437 (Khớp 100%)
2. Tính duy nhất relative_path : Duy nhất 100% (4437 path)
3. Phân bố 5 lớp bệnh:
   - ALGAL_LEAF_SPOT      :  733 ảnh (kỳ vọng:  733) -> KHỚP
   - ALLOCARIDARA_ATTACK  :  913 ảnh (kỳ vọng:  913) -> KHỚP
   - HEALTHY_LEAF         :  976 ảnh (kỳ vọng:  976) -> KHỚP
   - LEAF_BLIGHT          :  937 ảnh (kỳ vọng:  937) -> KHỚP
   - PHOMOPSIS_LEAF_SPOT  :  878 ảnh (kỳ vọng:  878) -> KHỚP
4. Nhóm gần trùng (Hamming <= 5): 4 nhóm, tổng 9 ảnh
   - Nhóm group_id=26  : 2 ảnh, lớp ALGAL_LEAF_SPOT
   - Nhóm group_id=517 : 3 ảnh, lớp HEALTHY_LEAF
   - Nhóm group_id=632 : 2 ảnh, lớp LEAF_BLIGHT
   - Nhóm group_id=1980: 2 ảnh, lớp ALLOCARIDARA_ATTACK
   -> Kiểm tra đồng nhất lớp   : 100% mỗi nhóm chỉ thuộc 1 lớp duy nhất
5. Đơn vị chia tập (group_key)  : 4432 key sẵn sàng cho split
✓ Toàn bộ kiểm tra dữ liệu đạt yêu cầu. Sẵn sàng sang bước chia tập.


---
## 2. Chia train/val/test

**Mục đích:** 
1. Phân chia tập dữ liệu DurianLDD theo tỉ lệ mục tiêu: 70% train, 10% val, 20% test, phân tầng (stratified) theo 5 lớp.
2. Đảm bảo toàn vẹn chống rò rỉ: mỗi nhóm gần trùng nằm trọn trong đúng MỘT tập duy nhất.
3. Phương pháp: Sử dụng `StratifiedGroupKFold` 2 bước:
   - **Bước 1:** `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)` lấy 1 fold làm test (~20% tổng số ảnh).
   - **Bước 2:** Trên phần còn lại (~80%), dùng `StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=SEED)` lấy 1 fold làm val (~10% tổng số ảnh, tức 1/8 của phần còn lại) và 7 fold làm train (~70% tổng số ảnh).
4. Thêm cột `split` vào dataframe.
5. Chuẩn bị dataframe cuối `df_final` với 4 cột chuẩn: `image_path` (đường dẫn tương đối), `class_name`, `group_id` (giữ nguyên giá trị gốc từ inventory), `split`.
6. Assert kiểm tra các ràng buộc và in bảng crosstab split × lớp cùng bảng đối soát tỉ lệ thực tế so với mục tiêu 70/10/20. *Chưa lưu file ở phần này.*

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf_test = StratifiedGroupKFold(n_splits=N_SPLITS_TEST, shuffle=True, random_state=SEED)
train_val_idx, test_idx = next(sgkf_test.split(
    X=df_durian,
    y=df_durian["class_name"],
    groups=df_durian["group_key"]
))

df_train_val = df_durian.iloc[train_val_idx].copy().reset_index(drop=True)
sgkf_val = StratifiedGroupKFold(n_splits=N_SPLITS_VAL, shuffle=True, random_state=SEED)
train_sub_idx, val_sub_idx = next(sgkf_val.split(
    X=df_train_val,
    y=df_train_val["class_name"],
    groups=df_train_val["group_key"]
))

train_idx = train_val_idx[train_sub_idx]
val_idx = train_val_idx[val_sub_idx]

df_durian["split"] = ""
df_durian.loc[train_idx, "split"] = "train"
df_durian.loc[val_idx, "split"] = "val"
df_durian.loc[test_idx, "split"] = "test"

df_final = df_durian.rename(columns={"relative_path": "image_path"})[
    ["image_path", "class_name", "group_id", "split"]
].copy()

print("Đã hoàn thành phân chia và chuẩn bị dataframe cuối df_final:")
print(f"  • Kích thước dataframe: {df_final.shape[0]} dòng, {df_final.shape[1]} cột")
print(f"  • Danh sách cột      : {df_final.columns.tolist()}\n")
print("5 dòng đầu của df_final:")
print(df_final.head(5))


Đã hoàn thành phân chia và chuẩn bị dataframe cuối df_final:
  • Kích thước dataframe: 4437 dòng, 4 cột
  • Danh sách cột      : ['image_path', 'class_name', 'group_id', 'split']

5 dòng đầu của df_final:
                                          image_path  ...  split
0  DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT...  ...  train
1  DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT...  ...  train
2  DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT...  ...  train
3  DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT...  ...   test
4  DLD_FinalDataset_224_spit/test/ALGAL_LEAF_SPOT...  ...  train

[5 rows x 4 columns]


In [ ]:
assert (df_final["split"] == "").sum() == 0, "Phát hiện ảnh chưa được gán tập split!"
assert set(df_final["split"].unique()) == {"train", "val", "test"}, "Tập hợp các split không hợp lệ!"
assert len(df_final) == EXPECTED_TOTAL_IMAGES, f"Tổng số ảnh bị thay đổi ({len(df_final)})!"
assert list(df_final.columns) == ["image_path", "class_name", "group_id", "split"], "Cấu trúc cột df_final không đúng!"

for gid in dup_group_ids:
    splits_in_group = df_final[df_final["group_id"] == gid]["split"].unique()
    assert len(splits_in_group) == 1, (
        f"RÒ RỈ DỮ LIỆU: Nhóm gần trùng group_id={gid} bị chia vào nhiều tập khác nhau: {splits_in_group}!"
    )

print("✓ Assert thành công:")
print("  • 100% ảnh đã được gán split hợp lệ trong df_final (không thiếu, không thừa).")
print("  • 100% nhóm gần trùng nằm trọn vẹn trong đúng 1 tập (không rò rỉ).")


✓ Assert thành công:
  • 100% ảnh đã được gán split hợp lệ trong df_final (không thiếu, không thừa).
  • 100% nhóm gần trùng nằm trọn vẹn trong đúng 1 tập (không rò rỉ).


In [ ]:
split_order = ["train", "val", "test"]
class_cols = sorted(df_final["class_name"].unique())

crosstab_df = pd.crosstab(
    df_final["split"],
    df_final["class_name"],
    margins=True,
    margins_name="Total"
).reindex(split_order + ["Total"])

print("=" * 85)
print("           BẢNG CROSSTAB: PHÂN BỐ SPLIT × LỚP BỆNH (CÓ TỔNG)")
print("=" * 85)
print(crosstab_df.to_string())
print()

split_counts = df_final["split"].value_counts().reindex(split_order)
total_imgs = len(df_final)
target_map = {"train": TRAIN_RATIO * 100, "val": VAL_RATIO * 100, "test": TEST_RATIO * 100}

comp_rows = []
for s in split_order:
    cnt = int(split_counts[s])
    actual_pct = (cnt / total_imgs) * 100
    target_pct = target_map[s]
    diff_pct = actual_pct - target_pct
    comp_rows.append({
        "Split": s,
        "Số ảnh": cnt,
        "Tỉ lệ thực tế (%)": f"{actual_pct:.2f}%",
        "Mục tiêu (%)": f"{target_pct:.1f}%",
        "Chênh lệch (%)": f"{diff_pct:+.2f}%"
    })

comp_df = pd.DataFrame(comp_rows)
print("=" * 85)
print("       BẢNG ĐỐI SOÁT TỈ LỆ THỰC TẾ SO VỚI MỤC TIÊU (70 / 10 / 20)")
print("=" * 85)
print(comp_df.to_string(index=False))
print("=" * 85)


           BẢNG CROSSTAB: PHÂN BỐ SPLIT × LỚP BỆNH (CÓ TỔNG)
class_name  ALGAL_LEAF_SPOT  ALLOCARIDARA_ATTACK  HEALTHY_LEAF  LEAF_BLIGHT  PHOMOPSIS_LEAF_SPOT  Total
split                                                                                                  
train                   513                  638           683          657                  614   3105
val                      74                   92            97           93                   88    444
test                    146                  183           196          187                  176    888
Total                   733                  913           976          937                  878   4437

       BẢNG ĐỐI SOÁT TỈ LỆ THỰC TẾ SO VỚI MỤC TIÊU (70 / 10 / 20)
Split  Số ảnh Tỉ lệ thực tế (%) Mục tiêu (%) Chênh lệch (%)
train    3105            69.98%        70.0%         -0.02%
  val     444            10.01%        10.0%         +0.01%
 test     888            20.01%        20.0%         +0.01%


In [ ]:
print("Phân bổ của 4 nhóm gần trùng vào các tập split:")
for gid in dup_group_ids:
    group_rows = df_final[df_final["group_id"] == gid]
    assigned_split = group_rows["split"].iloc[0]
    assigned_class = group_rows["class_name"].iloc[0]
    print(f"  • Nhóm group_id={gid:<4}: {len(group_rows)} ảnh | Lớp: {assigned_class:<20} | Nằm trọn trong: {assigned_split.upper()}")
print("\n✓ Hoàn thành phần 2. Đã chuẩn bị df_final, chưa lưu file ở phần này.")


Phân bổ của 4 nhóm gần trùng vào các tập split:
  • Nhóm group_id=26  : 2 ảnh | Lớp: ALGAL_LEAF_SPOT      | Nằm trọn trong: VAL
  • Nhóm group_id=517 : 3 ảnh | Lớp: HEALTHY_LEAF         | Nằm trọn trong: TEST
  • Nhóm group_id=632 : 2 ảnh | Lớp: LEAF_BLIGHT          | Nằm trọn trong: TRAIN
  • Nhóm group_id=1980: 2 ảnh | Lớp: ALLOCARIDARA_ATTACK  | Nằm trọn trong: TRAIN

✓ Hoàn thành phần 2. Đã chuẩn bị df_final, chưa lưu file ở phần này.


---
## 3. Kiểm chứng split

**Mục đích:** Kiểm chứng nghiêm ngặt toàn bộ 5 tiêu chí của dataframe phân chia `df_final` trước khi xuất file. Mỗi mục thực hiện kiểm tra `assert` và in rõ trạng thái `PASS` / `FAIL`:
1. **Mục 1:** Tổng đúng 4.437 ảnh, đường dẫn tương đối `image_path` không trùng lặp, mỗi ảnh thuộc đúng 1 tập.
2. **Mục 2:** Tỉ lệ tập xấp xỉ 70/10/20 (chênh lệch tối đa 1%); tỉ lệ từng lớp trong mỗi tập xấp xỉ tỉ lệ gốc (chênh lệch tối đa 2%).
3. **Mục 3:** Mỗi nhóm gần trùng (4 nhóm, tổng 9 ảnh, theo `group_id` gốc) chỉ nằm trong đúng 1 split duy nhất.
4. **Mục 4:** Tuyệt đối không có ảnh thuộc bộ dữ liệu Vietnamese durian leaf.
5. **Mục 5:** Chạy lại toàn bộ phép chia với cùng SEED=42, đối soát 1-1 với kết quả ban đầu: phải giống hệt 100%.

In [ ]:
try:
    assert len(df_final) == EXPECTED_TOTAL_IMAGES, f"Tổng số ảnh ({len(df_final)}) != {EXPECTED_TOTAL_IMAGES}"
    assert df_final["image_path"].is_unique, "Phát hiện image_path bị trùng lặp!"
    assert (df_final["split"] == "").sum() == 0, "Có ảnh chưa có nhãn split!"
    assert df_final["split"].isna().sum() == 0, "Có ảnh mang giá trị split NaN!"
    assert set(df_final["split"].unique()) == {"train", "val", "test"}, "Nhãn split không đúng bộ ba train/val/test!"
    assert list(df_final.columns) == ["image_path", "class_name", "group_id", "split"], "Cấu trúc cột df_final không khớp!"
    print(f"[PASS] Mục 1: Tổng {len(df_final):,} ảnh, image_path duy nhất 100%, mỗi ảnh thuộc đúng 1 tập.")
except AssertionError as e:
    print(f"[FAIL] Mục 1: {e}")
    raise


[PASS] Mục 1: Tổng 4,437 ảnh, image_path duy nhất 100%, mỗi ảnh thuộc đúng 1 tập.


In [ ]:
try:
    target_ratios = {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO}
    for s, target_r in target_ratios.items():
        actual_r = (df_final["split"] == s).mean()
        diff = abs(actual_r - target_r)
        assert diff <= MAX_SPLIT_RATIO_TOLERANCE, (
            f"Tỉ lệ tập '{s}' ({actual_r*100:.2f}%) lệch quá {MAX_SPLIT_RATIO_TOLERANCE*100:.1f}% so với mục tiêu ({target_r*100:.1f}%)! Lệch thực tế: {diff*100:.2f}%"
        )

    for s in ["train", "val", "test"]:
        split_df = df_final[df_final["split"] == s]
        for cname, expected_cnt in EXPECTED_CLASS_COUNTS.items():
            actual_class_r = (split_df["class_name"] == cname).mean()
            orig_class_r = expected_cnt / EXPECTED_TOTAL_IMAGES
            diff = abs(actual_class_r - orig_class_r)
            assert diff <= MAX_CLASS_RATIO_TOLERANCE, (
                f"Lớp '{cname}' trong tập '{s}' ({actual_class_r*100:.2f}%) lệch quá {MAX_CLASS_RATIO_TOLERANCE*100:.1f}% so với tỉ lệ gốc ({orig_class_r*100:.2f}%)! Lệch thực tế: {diff*100:.2f}%"
            )

    print(f"[PASS] Mục 2: Tỉ lệ tập khớp mục tiêu 70/10/20 (lệch tối đa <= {MAX_SPLIT_RATIO_TOLERANCE*100:.1f}%); tỉ lệ 5 lớp trong từng tập bám sát phân bố gốc (lệch tối đa <= {MAX_CLASS_RATIO_TOLERANCE*100:.1f}%).")
except AssertionError as e:
    print(f"[FAIL] Mục 2: {e}")
    raise


[PASS] Mục 2: Tỉ lệ tập khớp mục tiêu 70/10/20 (lệch tối đa <= 1.0%); tỉ lệ 5 lớp trong từng tập bám sát phân bố gốc (lệch tối đa <= 2.0%).


In [ ]:
try:
    assert len(dup_group_ids) == EXPECTED_NEAR_DUP_GROUPS, (
        f"Số nhóm gần trùng ({len(dup_group_ids)}) != {EXPECTED_NEAR_DUP_GROUPS}"
    )
    total_near_dup_imgs = len(df_final[df_final["group_id"].isin(dup_group_ids)])
    assert total_near_dup_imgs == EXPECTED_NEAR_DUP_IMAGES, (
        f"Tổng số ảnh gần trùng ({total_near_dup_imgs}) != {EXPECTED_NEAR_DUP_IMAGES}"
    )

    for gid in dup_group_ids:
        splits = df_final[df_final["group_id"] == gid]["split"].unique()
        assert len(splits) == 1, (
            f"RÒ RỈ DỮ LIỆU: Nhóm group_id={gid} bị phân chia sang {len(splits)} tập: {splits}!"
        )

    print(f"[PASS] Mục 3: Cả {EXPECTED_NEAR_DUP_GROUPS} nhóm gần trùng (tổng {EXPECTED_NEAR_DUP_IMAGES} ảnh, group_id gốc) đều nằm trọn vẹn trong đúng 1 split duy nhất (không rò rỉ).")
except AssertionError as e:
    print(f"[FAIL] Mục 3: {e}")
    raise


[PASS] Mục 3: Cả 4 nhóm gần trùng (tổng 9 ảnh, group_id gốc) đều nằm trọn vẹn trong đúng 1 split duy nhất (không rò rỉ).


In [ ]:
try:
    vn_mask = df_final["image_path"].str.lower().str.contains("vietnamese")
    assert vn_mask.sum() == 0, f"Phát hiện {vn_mask.sum()} ảnh Vietnamese durian leaf trong tập chia!"
    assert df_final["image_path"].str.startswith("DLD_FinalDataset_224_spit/").all(), (
        "Có ảnh không bắt đầu bằng thư mục chuẩn 'DLD_FinalDataset_224_spit/'!"
    )
    print("[PASS] Mục 4: Tuyệt đối không chứa ảnh thuộc Vietnamese durian leaf (0/4.437 ảnh).")
except AssertionError as e:
    print(f"[FAIL] Mục 4: {e}")
    raise


[PASS] Mục 4: Tuyệt đối không chứa ảnh thuộc Vietnamese durian leaf (0/4.437 ảnh).


In [ ]:
try:
    sgkf_test_re = StratifiedGroupKFold(n_splits=N_SPLITS_TEST, shuffle=True, random_state=SEED)
    train_val_idx_re, test_idx_re = next(sgkf_test_re.split(
        X=df_durian,
        y=df_durian["class_name"],
        groups=df_durian["group_key"]
))

    df_train_val_re = df_durian.iloc[train_val_idx_re].copy().reset_index(drop=True)
    sgkf_val_re = StratifiedGroupKFold(n_splits=N_SPLITS_VAL, shuffle=True, random_state=SEED)
    train_sub_idx_re, val_sub_idx_re = next(sgkf_val_re.split(
        X=df_train_val_re,
        y=df_train_val_re["class_name"],
        groups=df_train_val_re["group_key"]
    ))

    train_idx_re = train_val_idx_re[train_sub_idx_re]
    val_idx_re = train_val_idx_re[val_sub_idx_re]

    re_split = pd.Series(index=df_final.index, dtype=str)
    re_split.loc[train_idx_re] = "train"
    re_split.loc[val_idx_re] = "val"
    re_split.loc[test_idx_re] = "test"

    match_count = (df_final["split"] == re_split).sum()
    assert match_count == EXPECTED_TOTAL_IMAGES, (
        f"Phép chia lại không khớp 100% kết quả đầu ({match_count}/{EXPECTED_TOTAL_IMAGES} ảnh khớp)!"
    )
    print(f"[PASS] Mục 5: Tái tạo phép chia với cùng SEED={SEED} cho kết quả giống hệt 100% ({match_count}/{EXPECTED_TOTAL_IMAGES} ảnh khớp hoàn toàn).")
except AssertionError as e:
    print(f"[FAIL] Mục 5: {e}")
    raise


[PASS] Mục 5: Tái tạo phép chia với cùng SEED=42 cho kết quả giống hệt 100% (4437/4437 ảnh khớp hoàn toàn).


In [ ]:
print("=" * 75)
print("           TỔNG KẾT KẾT QUẢ KIỂM CHỨNG SPLIT (PHẦN 3)")
print("=" * 75)
print("  [PASS] 1. Tổng 4.437 ảnh, image_path không trùng, mỗi ảnh thuộc đúng 1 tập.")
print("  [PASS] 2. Tỉ lệ tập ≈ 70/10/20 (lệch <= 1%); tỉ lệ 5 lớp ≈ gốc (lệch <= 2%).")
print("  [PASS] 3. Mỗi nhóm gần trùng (4 nhóm, 9 ảnh, group_id gốc) chỉ nằm trong đúng 1 split.")
print("  [PASS] 4. Tuyệt đối không chứa ảnh Vietnamese durian leaf (0/4.437).")
print(f"  [PASS] 5. Tái lập 100% kết quả với SEED={SEED}.")
print("=" * 75)
print("✓ Toàn bộ 5/5 tiêu chí kiểm chứng đạt yêu cầu tuyệt đối trên df_final.")


           TỔNG KẾT KẾT QUẢ KIỂM CHỨNG SPLIT (PHẦN 3)
  [PASS] 1. Tổng 4.437 ảnh, image_path không trùng, mỗi ảnh thuộc đúng 1 tập.
  [PASS] 2. Tỉ lệ tập ≈ 70/10/20 (lệch <= 1%); tỉ lệ 5 lớp ≈ gốc (lệch <= 2%).
  [PASS] 3. Mỗi nhóm gần trùng (4 nhóm, 9 ảnh, group_id gốc) chỉ nằm trong đúng 1 split.
  [PASS] 4. Tuyệt đối không chứa ảnh Vietnamese durian leaf (0/4.437).
  [PASS] 5. Tái lập 100% kết quả với SEED=42.
✓ Toàn bộ 5/5 tiêu chí kiểm chứng đạt yêu cầu tuyệt đối trên df_final.


---
## 4. Lưu manifest và khóa test

**Mục đích:** 
1. Lưu tệp phân chia cố định `manifests/split_manifest.csv` với 4 cột chuẩn: `image_path` (đường dẫn tương đối), `class_name`, `group_id` (giữ giá trị gốc), `split`. Nếu file đã tồn tại thì dừng, không ghi đè.
2. Lưu tệp siêu dữ liệu `manifests/split_meta.json` lưu trữ: seed, tỉ lệ mục tiêu, phương pháp chia, số ảnh mỗi tập và mỗi lớp, ngày tạo, phiên bản thư viện (Python, Pandas, Scikit-learn), và mã băm SHA256 của tệp manifest.
3. Đặt cả hai tệp trên ở chế độ chỉ đọc (`read-only`) bằng `os.chmod` và kiểm tra lại bằng `os.access`.
4. Đảm bảo tài liệu giao thức `manifests/SPLIT_PROTOCOL.md` quy định nghiêm ngặt về việc không rò rỉ tập test.
5. Import hàm `get_split` từ module `src.data.splits` và chạy thử nghiệm tích hợp: xác nhận tải `train`, `val` thành công, kiểm tra cơ chế chặn truy cập `test` khi chưa có `FINAL_EVAL=1` và mở khóa khi có cờ `FINAL_EVAL=1`.

In [ ]:
if MANIFEST_PATH.exists():
    raise FileExistsError(
        f"Tệp manifest đã tồn tại tại '{MANIFEST_PATH}'! "
        "Dừng lại theo quy tắc an toàn, không ghi đè lên manifest đã tạo."
    )

df_final.to_csv(MANIFEST_PATH, index=False, encoding="utf-8")
print(f"✓ Đã lưu thành công manifest tại: {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
print(f"  • Kích thước: {len(df_final)} dòng ({len(df_final.columns)} cột: {df_final.columns.tolist()})")


✓ Đã lưu thành công manifest tại: manifests\split_manifest.csv
  • Kích thước: 4437 dòng (4 cột: ['image_path', 'class_name', 'group_id', 'split'])


In [ ]:
import datetime
import hashlib
import json
import platform
import sklearn

def compute_file_sha256(filepath: Path) -> str:
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

manifest_sha256 = compute_file_sha256(MANIFEST_PATH)

# Chuẩn bị siêu dữ liệu
meta_data = {
    "seed": SEED,
    "ratios": {
        "train": TRAIN_RATIO,
        "val": VAL_RATIO,
        "test": TEST_RATIO,
    },
    "method": (
        "2-stage StratifiedGroupKFold (n_splits=5 for test ~20%; "
        "n_splits=8 on remainder for val ~10% and train ~70%; "
        "groups=group_key to isolate near-duplicates)"
    ),
    "total_images": len(df_final),
    "split_counts": df_final["split"].value_counts().to_dict(),
    "class_counts": df_final["class_name"].value_counts().to_dict(),
    "split_class_distribution": pd.crosstab(df_final["split"], df_final["class_name"]).to_dict(orient="index"),
    "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "manifest_file": str(MANIFEST_PATH.relative_to(PROJECT_ROOT)),
    "manifest_sha256": manifest_sha256,
}

if META_PATH.exists():
    raise FileExistsError(
        f"Tệp meta đã tồn tại tại '{META_PATH}'! Dừng lại, không ghi đè."
    )

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta_data, f, indent=2, ensure_ascii=False)

print(f"✓ Đã lưu thành công siêu dữ liệu tại: {META_PATH.relative_to(PROJECT_ROOT)}")
print(f"  • SHA256 của manifest: {manifest_sha256}")


✓ Đã lưu thành công siêu dữ liệu tại: manifests\split_meta.json
  • SHA256 của manifest: 6294d4cb4e1b7974f4244a6fba6f0d5183b04dbb92ba924aaff23fceac88ff5e


In [ ]:
import os
import stat

for file_p in [MANIFEST_PATH, META_PATH]:
    current_mode = file_p.stat().st_mode
    file_p.chmod(current_mode & ~stat.S_IWRITE)
    is_writable = os.access(file_p, os.W_OK)
    assert not is_writable, f"Lỗi: Tệp {file_p.name} vẫn có quyền ghi!"
    print(f"✓ Đã khóa read-only: {file_p.relative_to(PROJECT_ROOT)} (os.access W_OK: {is_writable})")


✓ Đã khóa read-only: manifests\split_manifest.csv (os.access W_OK: False)
✓ Đã khóa read-only: manifests\split_meta.json (os.access W_OK: False)


In [ ]:
assert PROTOCOL_PATH.is_file(), f"Không tìm thấy file giao thức tại: {PROTOCOL_PATH}"
print(f"✓ Đã xác nhận giao thức phân chia: {PROTOCOL_PATH.relative_to(PROJECT_ROOT)}")


✓ Đã xác nhận giao thức phân chia: manifests\SPLIT_PROTOCOL.md


In [ ]:
import os
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.splits import get_split

df_loaded_train = get_split("train")
df_loaded_val = get_split("val")
assert len(df_loaded_train) == 3105, f"Số ảnh train ({len(df_loaded_train)}) != 3105"
assert len(df_loaded_val) == 444, f"Số ảnh val ({len(df_loaded_val)}) != 444"
print(f"✓ get_split('train'): tải thành công {len(df_loaded_train)} ảnh.")
print(f"✓ get_split('val')  : tải thành công {len(df_loaded_val)} ảnh.")

os.environ.pop("FINAL_EVAL", None)
try:
    get_split("test")
    assert False, "LỖI BẢO MẬT: get_split('test') không chặn khi chưa có FINAL_EVAL=1!"
except PermissionError as e:
    print(f"✓ Khóa test hoạt động chuẩn xác: Bắt được PermissionError ({e})")

os.environ["FINAL_EVAL"] = "1"
try:
    df_loaded_test = get_split("test")
    assert len(df_loaded_test) == 888, f"Số ảnh test ({len(df_loaded_test)}) != 888"
    print(f"✓ Mở khóa test thành công với FINAL_EVAL=1: tải được {len(df_loaded_test)} ảnh.")
finally:
    os.environ.pop("FINAL_EVAL", None)
    print("✓ Đã khóa lại cờ FINAL_EVAL sau thử nghiệm.")


✓ get_split('train'): tải thành công 3105 ảnh.
✓ get_split('val')  : tải thành công 444 ảnh.
✓ Khóa test hoạt động chuẩn xác: Bắt được PermissionError (Tập test đã bị khóa! Không được phép truy cập tập test trừ khi biến môi trường FINAL_EVAL=1.)
✓ Mở khóa test thành công với FINAL_EVAL=1: tải được 888 ảnh.
✓ Đã khóa lại cờ FINAL_EVAL sau thử nghiệm.


In [ ]:
print("=" * 75)
print("        HOÀN TẤT LƯU MANIFEST VÀ KHÓA BẢO VỆ TẬP TEST (PHẦN 4)")
print("=" * 75)
print(f"1. Manifest file    : {MANIFEST_PATH.relative_to(PROJECT_ROOT)} (Read-only, 4.437 ảnh)")
print(f"2. Metadata file    : {META_PATH.relative_to(PROJECT_ROOT)} (Read-only, SHA256 verified)")
print(f"3. Protocol file    : {PROTOCOL_PATH.relative_to(PROJECT_ROOT)}")
print(f"4. Mô-đun truy cập  : src.data.splits.get_split (chặn test trừ khi FINAL_EVAL=1)")
print("=" * 75)
print("✓ Toàn bộ quy trình phân chia dữ liệu DurianLDD đã hoàn thành an toàn.")


        HOÀN TẤT LƯU MANIFEST VÀ KHÓA BẢO VỆ TẬP TEST (PHẦN 4)
1. Manifest file    : manifests\split_manifest.csv (Read-only, 4.437 ảnh)
2. Metadata file    : manifests\split_meta.json (Read-only, SHA256 verified)
3. Protocol file    : manifests\SPLIT_PROTOCOL.md
4. Mô-đun truy cập  : src.data.splits.get_split (chặn test trừ khi FINAL_EVAL=1)
✓ Toàn bộ quy trình phân chia dữ liệu DurianLDD đã hoàn thành an toàn.
